# 02 - Inference Smoketest Pipeline
## 1. Pipeline Objective

## 2. Dependency Management
### 2.1 Package Installation

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas matplotlib seaborn


### 2.2 Module Imports

In [ ]:
import os
import json
import gc
import re
import time
from typing import List, Dict, Any

import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

### 2.3 Environment Variables

In [ ]:
# Global Configuration
PROJECT_ROOT: str = '/content/drive/MyDrive/LLMReliability'
DATA_PROCESSED_DIR: str = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR: str = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR: str = os.path.join(PROJECT_ROOT, 'results')
MODEL_ID: str = "Qwen/Qwen3-4B-Instruct-2507"

# HF_HOME diarahkan ke persistent storage
os.environ['HF_HOME'] = MODELS_DIR

print(f"[MODEL] id='{MODEL_ID}' | status=CONFIGURING")

## 3. Resource Allocation
### 3.1 Identity & Access Management (IAM)

In [ ]:

try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    secrets_path: str = '/content/drive/MyDrive/secrets.json'
    if os.path.exists(secrets_path):
        with open(secrets_path, 'r') as f:
            secrets = json.load(f)
        hf_token = secrets.get("HF_TOKEN", "")
        if hf_token:
            from huggingface_hub import login
            login(token=hf_token)
            print("[ENVIRONMENT] status=AUTHENTICATED | component='HuggingFace'")
except ImportError:
    print("[ENVIRONMENT] status=LOCAL | component='Google Drive'")

### 3.2 Storage Initialization

In [ ]:
for d in [DATA_PROCESSED_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

## 4. Pilot Data Ingestion

In [ ]:
pilot_path = os.path.join(DATA_PROCESSED_DIR, 'indopref_pilot.parquet')
try:
    df_pilot = pd.read_parquet(pilot_path)
    df_smoketest = df_pilot.head(10).copy()
    print(f"[DATASET] split='smoketest' | samples={len(df_smoketest)} | status=LOADED")
except FileNotFoundError:
    print(f"[DATASET] split='pilot' | status=NOT_FOUND | path='{pilot_path}'")

## 5. Prompt Formatting Node

In [ ]:
def format_prompt_P1_AB(input_text: str, resp_A: str, resp_B: str) -> str:
    return (
        f"### TUGAS\nBandingkan dua respons AI di bawah ini untuk instruksi yang sama. Pilih mana yang secara keseluruhan lebih baik.\n\n"
        f"### INSTRUKSI\n{input_text}\n\n"
        f"### RESPONS A\n{resp_A}\n\n"
        f"### RESPONS B\n{resp_B}\n\n"
        f"### FORMAT JAWABAN\nKeluarkan HANYA satu huruf: A atau B. Dilarang menulis alasan atau teks lain."
    )

def apply_chat_template(tokenizer, text: str) -> str:
    messages = [
        {"role": "system", "content": "Anda adalah evaluator AI yang objektif. Anda tidak memiliki bias posisi. Jawab HANYA dengan pilihan kandidat."},
        {"role": "user", "content": text}
    ]
    return tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )


## 6. LLM Initialization (Quantized)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=MODELS_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=MODELS_DIR
)
allocated_vram = torch.cuda.memory_allocated() / 1024**3
print(f"[MODEL] status=LOADED | vram='{allocated_vram:.2f} GB'")

## 7. Batched Execution Engine

In [ ]:
smoketest_results_path = os.path.join(RESULTS_DIR, 'smoketest_outputs.jsonl')
results: List[Dict[str, Any]] = []

print(
    f"[INFERENCE] target='smoketest' | "
    f"samples={len(df_smoketest)} | "
    f"status=STARTED"
)

# Optimization: Inference Mode prevents PyTorch from storing computational graphs (Saves Memory)
with torch.inference_mode():
    for index, row in df_smoketest.iterrows():
        start_time = time.time()
        
        # 1. Prepare Content
        raw_prompt = format_prompt_P1_AB(row['input'], row['response_1'], row['response_2'])
        formatted_prompt = apply_chat_template(tokenizer, raw_prompt)
        
        # 2. Tokenize
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        
        # 3. Generate (Deterministic & Short)
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
        # 4. Decode & TPS Calc
        input_len = inputs['input_ids'].shape[1]
        generated_tokens = outputs[0][input_len:]
        raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        runtime = time.time() - start_time
        output_len = len(generated_tokens)
        tps = output_len / runtime if runtime > 0 else 0.0
        
        # 5. Compile Result
        record = {
            "sample_id": int(row['original_id']),
            "domain": row['domain'],
            "prompt_id": "P1",
            "position": "AB",
            "human_preference": "A" if row['chosen_model'] == row['response_1'] else "B",
            "raw_output": raw_output,
            "runtime_sec": round(runtime, 2),
            "output_tokens": output_len,
            "tps": round(tps, 2)
        }
        results.append(record)
        
        # Incremental Save
        with open(smoketest_results_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record) + '\n')
            
        print(
            f"[INFERENCE] sample_id={row['original_id']} | "
            f"output='{raw_output}' | "
            f"tps={tps:.2f} | "
            f"runtime={runtime:.2f}s"
        )
        
        # 6. Garbage Collection (OOM Mitigation)
        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()

print(
    f"[INFERENCE] status=COMPLETED | "
    f"path='{smoketest_results_path}'"
)

## 8. Response Parser & Telemetry
### 8.1 Semantic Extraction (Regex)

In [ ]:
def parse_decision(raw_output: str) -> str:
    clean_text = raw_output.strip().upper()
    match = re.search(r'\b([AB])\b', clean_text)
    if match:
        return match.group(1)
    return "INVALID"
valid_count = 0
invalid_count = 0

for r in results:
    parsed = parse_decision(r["raw_output"])
    r["parsed_decision"] = parsed
    if parsed in ["A", "B"]:
        valid_count += 1
    else:
        invalid_count += 1

# Create DataFrame for Observability
df_results = pd.DataFrame(results)

display(df_results[['sample_id', 'domain', 'human_preference', 'raw_output', 'parsed_decision']])

### 8.2 Latency Monitoring

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_results['runtime_sec'].plot(kind='bar', color='#3498db', ax=axes[0])
axes[0].set_title('Integritas Performa: Latensi per Sampel (Detik)')
axes[0].set_xlabel('Indeks Sampel')
axes[0].set_ylabel('Detik')

df_results['tps'].plot(kind='line', marker='o', color='#e74c3c', ax=axes[1])
axes[1].set_title('Integritas Performa: Tokens Per Second (TPS)')
axes[1].set_xlabel('Indeks Sampel')
axes[1].set_ylabel('TPS')

plt.tight_layout()
plt.show()

### 8.3 Metric Aggregation

In [ ]:
invalid_rate = (invalid_count / len(results)) * 100
avg_runtime = df_results["runtime_sec"].mean()
avg_tps = df_results["tps"].mean()

print(
    f"[METRICS] total={len(results)} | "
    f"valid={valid_count} | "
    f"invalid={invalid_count} | "
    f"failure_rate={invalid_rate:.1f}%"
)
print(
    f"[METRICS] avg_runtime={avg_runtime:.2f}s | "
    f"avg_tps={avg_tps:.2f}"
)

assert invalid_rate < 20.0, (
    "[INFERENCE] status=FAILED | "
    "error='parse_failure_rate_exceeded' | "
    f"actual={invalid_rate:.1f}% | "
    "threshold=20.0%"
)